<a href="https://colab.research.google.com/github/piyushnanwani/RAG-system-building-challenge/blob/main/3_May_RAGSystemBuildingChallenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-generativeai chromadb sentence-transformers pypdf

In [ ]:
from google import genai
from google.genai import types

# 1. Initialize the Client
client = genai.Client(api_key="")

# 2. Use the correct Preview model ID
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=0.95
    ),
    contents="Testing connection: Respond with 'System Ready'"
)

print(response.text)

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb

# 1. Load and Chunk with Metadata
reader = PdfReader("SBI_Contract.pdf")
chunks = []
metadatas = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    # Splitting pages into 2 halves for better search granularity
    mid = len(text) // 2
    chunks.extend([text[:mid], text[mid:]])
    metadatas.extend([{"page": i+1}, {"page": i+1}])

# 2. Initialize 2026 Persistent Database
# Using PersistentClient so the data stays in the /content/ folder
chroma_client = chromadb.PersistentClient(path="./sbi_contract_db")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

collection = chroma_client.get_or_create_collection(name="sbi_it_contract")

# 3. Index with Metadata
for i, chunk in enumerate(chunks):
    vector = embed_model.encode(chunk).tolist()
    collection.add(
        ids=[f"chunk_{i}"],
        embeddings=[vector],
        documents=[chunk],
        metadatas=[metadatas[i]]
    )

In [ ]:
def query_contract(user_query):
    # 1. Similarity Search (Fetch top 5 results for complex contracts)
    query_vector = embed_model.encode(user_query).tolist()
    results = collection.query(query_embeddings=[query_vector], n_results=5)

    # 2. Format Context with Page Numbers
    context_list = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        context_list.append(f"[Page {meta['page']}]: {doc}")

    context = "\n---\n".join(context_list)

    # 3. Execution
    full_prompt = f"Using the following sections from the SBI Contract, answer: {user_query}\n\nCONTEXT:\n{context}"

    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        config=legal_config,
        contents=full_prompt
    )
    return response.text

# Test a high-stakes question
print(query_contract("What are the specific financial penalties for a Tier 1 system failure?"))

In [ ]:
# Configuration for Legal Accuracy
legal_config = types.GenerateContentConfig(
    # REMOVED: model="gemini-3-flash-preview",
    temperature=0.0,  # CRITICAL: 0.0 for zero creativity in legal work
    top_p=1,
    system_instruction="You are a senior legal auditor. Answer based ONLY on the provided contract clauses."
)